In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import jax
jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", True)

In [ ]:
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import numpy as np
import jax.numpy as jnp
import equinox as eqx

from rhmag.utils.pretest_evaluation import create_multilevel_df
from rhmag.model_interfaces.model_interface import ModelInterface

from rhmag.data_management import FINAL_MATERIALS, MaterialSet, DataSet
from rhmag.utils.data_plotting import plot_sequence_prediction, plot_hysteresis_prediction
from rhmag.utils.model_evaluation import reconstruct_model_from_file, plot_model_frequency_sweep, evaluate_cross_validation, get_exp_ids
from rhmag.utils.final_data_evaluation import (
    FINAL_MATERIALS,
    TestSet,
    ResultSet,
    predict_test_scenarios,
    validate_result_set,
    visualize_result_set,
    evaluate_test_scenarios,
    update_pareto_df,
    get_exp_ids_per_material,
    predict_test_scenarios_single_material
)
from rhmag.model_setup import setup_normalizer, setup_dataset

In [ ]:
test_data_per_material = {material_name: TestSet.from_material_name(material_name) for material_name in FINAL_MATERIALS}

#### Quantitative Comparison:

In [ ]:
adapted_RMS_results_df = update_pareto_df(
    pareto_results_path=None,
    exp_ids_per_material=get_exp_ids_per_material(
        model_type="GRU8",
        exp_name="ablation-default-f32",
    ),
    test_data_per_material=test_data_per_material,
)

In [ ]:
adapted_RMS_only_seq_results_df = update_pareto_df(
    pareto_results_path=None,
    exp_ids_per_material=get_exp_ids_per_material(
        model_type="GRU8",
        exp_name="ablation-loss-function-only-seq-weighting-f32",
    ),
    test_data_per_material=test_data_per_material,
)

In [ ]:
adapted_RMS_only_B_results_df = update_pareto_df(
    pareto_results_path=None,
    exp_ids_per_material=get_exp_ids_per_material(
        model_type="GRU8",
        exp_name="ablation-loss-function-only-delta-B-f32",
    ),
    test_data_per_material=test_data_per_material,
)

In [ ]:
MSE_results_df = update_pareto_df(
    pareto_results_path=None,
    exp_ids_per_material=get_exp_ids_per_material(
        model_type="GRU8",
        exp_name= "ablation-loss-function-f32",
    ),
    test_data_per_material=test_data_per_material,
)

---

In [ ]:
# compare the results

display("adapted_RMS:", adapted_RMS_results_df.groupby("material").mean(numeric_only=True))
display("MSE:", MSE_results_df.groupby("material").mean(numeric_only=True))

In [ ]:
import pandas as pd
import seaborn as sns

In [ ]:
material_name_map = {
    "A": "3C92",
    "B": "3C95",
    "C": "FEC007",
    "D": "FEC014",
    "E": "T37",
}

In [ ]:
# combine into a single df:

adapted_RMS_results_df["training_loss"] = "adapted RMS"
MSE_results_df["training_loss"] = "MSE"

adapted_RMS_only_B_results_df["training_loss"] = "adapted RMS (only $H^{-1}_\\mathrm{RMS}$)"
adapted_RMS_only_seq_results_df["training_loss"] = "adapted RMS (only $\Delta B$)"

full_results_df = pd.concat([
    adapted_RMS_results_df,
    adapted_RMS_only_B_results_df,
    adapted_RMS_only_seq_results_df,
    MSE_results_df,
], ignore_index=True)

full_results_df["material"] = full_results_df["material"].map(material_name_map)
full_results_df

In [ ]:
import matplotlib as mpl
from matplotlib import rc
from matplotlib.ticker import ScalarFormatter, StrMethodFormatter, LogLocator
import matplotlib.ticker as ticker

rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
def plot_ablation_comparison(full_results_df):
    fig, axs = plt.subplots(1, 4, figsize=(7.167, 7.167 / 3), constrained_layout=True)
    metric_label_map = {
        "sre_avg": r"$\mathrm{SRE}_{\mathrm{avg}}$", 
        "sre_95th": r"$\mathrm{SRE}_{95\mathrm{-th}}$",
        "nere_avg": r"$\mathrm{NERE}_{\mathrm{avg}}$",
        "nere_95th": r"$\mathrm{NERE}_{95\mathrm{-th}}$",
    }
    
    for idx, (ax, metric) in enumerate(zip(axs, ["sre_avg", "sre_95th", "nere_avg", "nere_95th"])):
        sns.stripplot(
            data=full_results_df,
            x="material",
            y=metric,
            hue="training_loss",
            palette={
                "adapted RMS": "tab:blue",
                "adapted RMS (only $H^{-1}_\\mathrm{RMS}$)": "tab:red",
                "adapted RMS (only $\Delta B$)": "tab:purple",
                "MSE": "tab:orange"},
            legend=True if idx==0 else False,
            ax=ax,
            size=3.5,
            alpha=0.8,
            #marker="x",
            dodge=True,
        )
        ax.tick_params(axis='x', labelrotation=90)
        ax.set_ylabel(metric_label_map[metric])
        ax.set_yscale('log')
        ax.tick_params(which="both", axis="y", direction="in")
        ax.tick_params(which="both", axis="x", direction="in")

        ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.3f}'))
        ax.yaxis.set_minor_formatter(StrMethodFormatter('{x:.3f}'))
        ax.yaxis.set_major_locator(ticker.LogLocator(numticks=5))
        ax.yaxis.set_minor_locator(ticker.LogLocator(subs=np.arange(2, 10, 2) * 0.1, numticks=3))

        ax.grid(True, which="both", alpha=0.3)

        num_categories = full_results_df['material'].nunique()
        for i in range(1, num_categories, 2):
            ax.axvspan(i - 0.5, i + 0.5, color='gray', alpha=0.15) 


    handles, labels = axs[0].get_legend_handles_labels()

    print(handles, labels)
    fig.legend(
        handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=4
    )
    axs[0].legend().remove()

    return fig, axs

In [ ]:
fig, axs = plot_ablation_comparison(full_results_df)

plt.savefig("ablation_study_loss_quantitative.pdf", bbox_inches="tight")
plt.show()

In [ ]:
raise

#### Qualitative comparison:

Can you actually see any difference in the qualitative trajectories?

visualize some exemplary trajectories from the data sets

In [ ]:
def plot_timeseries(H_past, H_future, B_past, B_future, H_pred, max_n_length=None, figsize=None):
    H_full_true = jnp.concatenate([H_past, H_future])
    B_full_true = jnp.concatenate([B_past, B_future])
    H_full_pred = jnp.concatenate([H_past, H_pred])
    if max_n_length is not None:
        H_full_true = H_full_true[:max_n_length]
        B_full_true = B_full_true[:max_n_length]
        H_full_pred = H_full_pred[:max_n_length]

    tau = 1 / (16)
    t = np.linspace(0, (H_full_true.shape[0] -1) * tau, H_full_true.shape[0])
    
    fig, axs = plt.subplots(1,1, figsize=figsize)
    axs.plot(
        t,
        H_full_true,
        color="tab:blue",
        label="$H$",
    )
    axs.plot(
        t,
        H_full_pred,
        color="tab:orange",
        label="$\hat{H}$",
        linestyle="dashed",
    )
    axs.set_ylabel("$H \mathrm{\; in \; A/m}$")
    axs.set_xlabel("$t \mathrm{\; in \; \\upmu s}$")
    axs.grid(True, alpha=0.3)
    axs.tick_params(which="major", axis="y", direction="in")
    axs.tick_params(which="both", axis="x", direction="in")
    axs.legend()
    fig.tight_layout()
    return fig, axs


def plot_BH_curve_with_loss(H_past, H_future, B_past, B_future, H_pred, max_n_length=None, figsize=None):

    H_full_true = jnp.concatenate([H_past, H_future])
    B_full_true = jnp.concatenate([B_past, B_future])
    H_full_pred = jnp.concatenate([H_past, H_pred])
    if max_n_length is not None:
        H_full_true = H_full_true[:max_n_length]
        B_full_true = B_full_true[:max_n_length]
        H_full_pred = H_full_pred[:max_n_length]
    
    fig, axs = plt.subplots(1,1, figsize=figsize)#(7.167/2,7.167/2))
    axs.plot(
        H_full_true,
        B_full_true,
        color="tab:blue",
        label="$BH$",
    )
    axs.plot(
        H_full_pred,
        B_full_true,
        color="tab:orange",
        label="$B\\hat{H}$",
        linestyle="dashed",
    )

    axs.set_xlabel("$H \mathrm{\; in \; A/m}$")
    axs.set_ylabel("$B \mathrm{\; in \; T}$")
    axs.tick_params(which="major", axis="y", direction="in")
    axs.tick_params(which="both", axis="x", direction="in")
    axs.legend()
    axs.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig, axs

In [ ]:
material_name = "A"
model_idx = 0

test_set = test_data_per_material[material_name]

In [ ]:
test_set

In [ ]:
def load_models_per_material(model_type, exp_name):
    exp_ids_per_material = get_exp_ids_per_material(
        model_type=model_type,
        exp_name=exp_name,
    )
    models_per_material = {}
    for material_name, exp_ids in exp_ids_per_material.items():
        models_per_material[material_name] = [reconstruct_model_from_file(exp_id) for exp_id in exp_ids]
    
    return models_per_material

In [ ]:
models_per_material_MSE = load_models_per_material(
    model_type="GRU8", exp_name= "ablation-loss-function-f32"
)

models_per_material_RMS = load_models_per_material(
    model_type="GRU8", exp_name="ablation-default-f32",
)

In [ ]:
def predict_test_scenarios_inputs_per_material(
    models_per_material: dict[str, list[ModelInterface]],
    test_set_per_material: dict[str, TestSet],
):
    all_results_sets_per_material = {}
    for material_name in models_per_material.keys():
        models = models_per_material[material_name]
        test_set = test_set_per_material[material_name]

        if len(models) > 0:
            result_sets = []
            for model in models:
                result_set = predict_test_scenarios_single_material(model, test_set, None, material_name)
                result_sets.append(result_set)
    
            stacked_result_sets = ResultSet(
                H=jnp.stack([result_set.H for result_set in result_sets]),
                B=jnp.stack([result_set.B for result_set in result_sets]),
                T=jnp.stack([result_set.T for result_set in result_sets]),
                exp_id=None,
                material_name = result_sets[0].material_name,
            )
            all_results_sets_per_material[material_name] = stacked_result_sets

    return all_results_sets_per_material

In [ ]:
result_sets_MSE = predict_test_scenarios_inputs_per_material(models_per_material_MSE, test_data_per_material)

In [ ]:
result_sets_RMS = predict_test_scenarios_inputs_per_material(models_per_material_RMS, test_data_per_material)

compare some plots:

In [ ]:
def visualize_exemplary_prediction(
    stacked_result_sets: dict[str, ResultSet],
    test_set: ResultSet,
    seq_idx: int, 
    max_length: None | int = None,
    figsize: tuple[float] = (7.167, 7.167 / 3),
):
    color_cycle = plt.rcParams['axes.prop_cycle']
    colors = color_cycle.by_key()['color']
    
    tau = 1 / 16
    if max_length != None:
        trajectory_length = min(max_length, test_set.H.shape[-1])
    else:
        trajectory_length = test_set.H.shape[-1]

    t = jnp.linspace(0, trajectory_length - 1 * tau, trajectory_length)
    
    fig, axs = plt.subplots(1,1, figsize=figsize)
    
    
    plot_H_gt = test_set.H_gt[seq_idx, :trajectory_length]


    axs.plot(t, plot_H_gt, color="k", label="$H_{\mathrm{gt}}$", linestyle="dashed")

    for (key, result_set), color in zip(stacked_result_sets.items(), colors):
        plot_H = result_set.H[:, seq_idx, :trajectory_length]
        # for i in range(plot_H.shape[0]):
        #      axs.plot(t, plot_H[i], color=color, label=key, alpha=0.4)
        
        mean = jnp.mean(plot_H, axis=0)
        std = jnp.std(plot_H, axis=0)

        axs.plot(t, mean, color=color, label=key, alpha=0.8)
        axs.fill_between(t, mean - std, mean + std, color=color, alpha=0.2)
    
    axs.set_ylabel("$H \mathrm{\; in \; A/m}$")
    axs.set_xlabel("$t \mathrm{\; in \; \\upmu s}$")
    axs.grid(True, alpha=0.3)
    axs.tick_params(which="major", axis="y", direction="in")
    axs.tick_params(which="both", axis="x", direction="in")
    axs.legend()
    fig.tight_layout()
    return fig, axs


def visualize_exemplar_BH_loop(
    stacked_result_sets: dict[str, ResultSet],
    test_set: ResultSet,
    seq_idx: int, 
    max_length: None | int = None,
    figsize: tuple[float] = (7.167, 7.167 / 3),
):
    color_cycle = plt.rcParams['axes.prop_cycle']
    colors = color_cycle.by_key()['color']
    
    tau = 1 / 16
    if max_length != None:
        trajectory_length = min(max_length, test_set.H.shape[-1])
    else:
        trajectory_length = test_set.H.shape[-1]
    
    fig, axs = plt.subplots(1,1, figsize=figsize)
    
    plot_B = test_set.B[seq_idx, :trajectory_length]
    plot_H_gt = test_set.H_gt[seq_idx, :trajectory_length]


    axs.plot(plot_B, plot_H_gt, color="k", label="$H_{\mathrm{gt}}$", linestyle="dashed")

    for (key, result_set), color in zip(stacked_result_sets.items(), colors):
        plot_H = result_set.H[:, seq_idx, :trajectory_length]
        # for i in range(plot_H.shape[0]):
        #      axs.plot(plot_B, plot_H[i], color=color, label=key, alpha=0.1)
        
        mean = jnp.mean(plot_H, axis=0)
        std = jnp.std(plot_H, axis=0)

        axs.plot(plot_B, mean, color=color, label=key, alpha=0.8)
        #axs.fill_between(plot_B, mean - std, mean + std, color=color, alpha=0.2)
    
    axs.set_ylabel("$H \mathrm{\; in \; A/m}$")
    axs.set_xlabel("$B \mathrm{\; in \; Vs}$")
    axs.grid(True, alpha=0.3)
    axs.tick_params(which="major", axis="y", direction="in")
    axs.tick_params(which="both", axis="x", direction="in")
    axs.legend()
    fig.tight_layout()
    return fig, axs

In [ ]:
material_name = "C"

for seq_idx in range(5):

    visualize_exemplary_prediction(
        stacked_result_sets={
            "rms": result_sets_RMS[material_name],
            "mse": result_sets_MSE[material_name],
        },
        test_set=test_data_per_material[material_name],
        seq_idx=seq_idx,
        max_length=None,
        figsize=(30, 10),
    )
    plt.show()

In [ ]:
material_name = "C"

for seq_idx in range(5):

    visualize_exemplar_BH_loop(
        stacked_result_sets={
            "rms": result_sets_RMS[material_name],
            "mse": result_sets_MSE[material_name],
        },
        test_set=test_data_per_material[material_name],
        seq_idx=seq_idx,
        max_length=None,
        figsize=(30, 10),
    )
    plt.show()

In [ ]:
raise

In [ ]:
model_MSE = reconstruct_model_from_file(
    filename=get_exp_ids_per_material(
        model_type="GRU8",
        exp_name= "ablation-loss-function-f32",
    )[material_name][model_idx]
)

model_RMS = reconstruct_model_from_file(
    filename=get_exp_ids_per_material(
        model_type="GRU8",
        exp_name= "ablation-default-f32",
    )[material_name][model_idx]
)

In [ ]:
past_size = 100
frequency = 125_000

H_future=test_set.at_frequency(frequency).H[:, past_size:]
B_past=test_set.at_frequency(frequency).B[:, :past_size]
H_past=test_set.at_frequency(frequency).H[:, :past_size]
B_future=test_set.at_frequency(frequency).B[:, past_size:]
T=test_set.at_frequency(frequency).T

H_pred_MSE = model_MSE(
    B_past=B_past,
    H_past=H_past,
    B_future=B_future,
    T=T,
)

H_pred_RMS = model_RMS(
    B_past=B_past,
    H_past=H_past,
    B_future=B_future,
    T=T,
)

idx = 0
max_n_length = 900
fig, axs = plot_timeseries(
    H_past[idx],
    H_future[idx],
    B_past[idx],
    B_future[idx],
    H_pred_MSE[idx],
    max_n_length=max_n_length,
    figsize=(7.167/2,7.167/3),
)

H_full_true = jnp.concatenate([H_past[idx], H_future[idx]])
H_full_pred = jnp.concatenate([H_past[idx], H_pred_RMS[idx]])
H_full_true = H_full_true[:max_n_length]
H_full_pred = H_full_pred[:max_n_length]

tau = 1 / (16)
t = np.linspace(0, (H_full_true.shape[0] -1) * tau, H_full_true.shape[0])
axs.plot(t, H_full_pred, color="tab:purple", linestyle="dotted", label="$\hat{H}_\mathrm{RMS}$",)
axs.legend()

plt.savefig("ablation_loss_qualitative.pdf")

plt.show()

# fig, axs = plot_BH_curve_with_loss(
#     H_past[idx],
#     H_future[idx],
#     B_past[idx],
#     B_future[idx],
#     H_pred_MSE[idx],
#     max_n_length=None,
#     figsize=(7.167/2,7.167/3),
# )
# plt.show()

In [ ]:
raise

- Maybe it makes sense to show mean +- std of the models for the simulation trajectory

In [ ]:
# plot the second model's prediction into the same figure:
raise NotImplementedError


for scenario in test_set.scenarios:
    H_pred = model(
        B_past=scenario.B_past,
        H_past=scenario.H_past,
        B_future=scenario.B_future,
        T=jnp.squeeze(scenario.T),
    )

    B_future = scenario.B_future
    H_future = scenario.H_future
    B_past = scenario.B_past

    start_idx = 0
    n_plots = 5

    for start_idx in np.arange(0, H_pred.shape[0], n_plots):
    
        fig, axs = plt.subplots(3, n_plots, figsize=(12,7))
        for idx in range(n_plots):
            axs[0, idx].plot(B_future[start_idx+idx])
            axs[1, idx].plot(H_future[start_idx+idx])
            axs[1, idx].plot(H_pred[start_idx+idx])
            axs[1, idx].plot(H_future[start_idx+idx] - H_pred[start_idx+idx], color="tab:red", linestyle="--")
        
            axs[2, idx].plot(B_future[start_idx+idx], H_future[start_idx+idx])
            axs[2, idx].plot(B_future[start_idx+idx], H_pred[start_idx+idx])
        
            axs[0, idx].grid(True, alpha=0.3)
            axs[1, idx].grid(True, alpha=0.3)
            axs[2, idx].grid(True, alpha=0.3)
        
            axs[0, idx].set_ylabel("B")
            axs[0, idx].set_xlabel("k")
            axs[1, idx].set_ylabel("H")
            axs[1, idx].set_xlabel("k")
            axs[2, idx].set_ylabel("H")
            axs[2, idx].set_xlabel("B")
        
        fig.tight_layout(pad=-0.2)
        plt.show()